In [ ]:
import os
import io
import re
from google.cloud import vision
from pdf2image import convert_from_path

# --- CONFIGURACIÓN DE RUTAS ---
nombre_archivo_llave = "" 
ruta_descargas = os.path.join(os.path.expanduser("~"), "Downloads", nombre_archivo_llave)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = ruta_descargas

POPPLER_PATH = r''
DIR_PDFS = r'' 
client = vision.ImageAnnotatorClient()

def extraer_folio_puro_8_digitos(path_pdf):
    # Intentamos en orden de probabilidad: pág 1 (frente), 3 (anexo), 2 (reverso)
    paginas_objetivo = [1, 3, 2]
    
    for num_pag in paginas_objetivo:
        try:
            # Usamos 400 DPI para que los números pequeños del folio no se confundan con letras
            images = convert_from_path(
                path_pdf, 
                first_page=num_pag, 
                last_page=num_pag, 
                poppler_path=POPPLER_PATH,
                dpi=400 
            )
            
            if images:
                img_byte_arr = io.BytesIO()
                images[0].save(img_byte_arr, format='JPEG', quality=95)
                content = img_byte_arr.getvalue()
                
                image = vision.Image(content=content)
                response = client.text_detection(image=image)
                
                if not response.text_annotations:
                    continue

                texto = response.text_annotations[0].description
                
                # --- FILTRO DE SEGURIDAD ---
                # 1. Buscamos cadenas de exactamente 8 números (\b evita que agarre partes de la CURP)
                # 2. Nos aseguramos que NO sean los mismos del CP (aunque el CP tiene 5, esto es extra seguridad)
                match = re.search(r'\b(\d{8})\b', texto)
                
                if match:
                    folio_detectado = match.group(1)
                    # Opcional: Si sabes que el folio NUNCA empieza con cierto número, puedes filtrarlo aquí
                    return folio_detectado
                    
        except Exception:
            continue
            
    return None

def proceso_renombrado_limpio():
    if not os.path.exists(DIR_PDFS):
        print(f"❌ Carpeta no encontrada: {DIR_PDFS}")
        return

    print(f"🚀 Iniciando limpieza y renombrado (SOLO FOLIOS DE 8 DÍGITOS)...")

    # Filtramos archivos que NO tengan el formato correcto todavía
    # OJO: He incluido que procese los que dicen "Licencia_Casado" para corregirlos
    archivos = [f for f in os.listdir(DIR_PDFS) if f.lower().endswith(".pdf")]

    exitos = 0
    fallos = 0

    for filename in archivos:
        # Si el archivo ya tiene 8 números después de "Licencia_", lo saltamos para ahorrar tiempo
        if re.search(r'Licencia_\d{8}\.pdf', filename):
            continue

        path_completo = os.path.join(DIR_PDFS, filename)
        folio = extraer_folio_puro_8_digitos(path_completo)
        
        if folio:
            nuevo_nombre = f"Licencia_{folio}.pdf"
            path_destino = os.path.join(DIR_PDFS, nuevo_nombre)
            
            try:
                # Manejo de duplicados para no borrar nada
                if os.path.exists(path_destino) and path_completo != path_destino:
                    path_destino = path_destino.replace(".pdf", f"_REVISAR_{exitos}.pdf")

                os.rename(path_completo, path_destino)
                print(f"✅ CORREGIDO: {filename} -> {nuevo_nombre}")
                exitos += 1
            except Exception as e:
                print(f"❌ Error en {filename}: {e}")
        else:
            # Solo avisar de los que realmente no tienen nada parecido a un folio
            if "Licencia_" in filename:
                print(f"💀 No se pudo extraer folio numérico de: {filename}")
                fallos += 1

    print("\n" + "="*40)
    print(f"✨ RESULTADOS FINALES")
    print(f"✔️ Archivos corregidos/renombrados: {exitos}")
    print(f"❌ Archivos que requieren revisión manual: {fallos}")
    print("="*40)

if __name__ == "__main__":
    proceso_renombrado_limpio()

In [ ]:
import os
import io
import re
import pandas as pd
from google.cloud import vision
from pdf2image import convert_from_path

# --- 1. CONFIGURACIÓN DE ENTORNO ---
nombre_archivo_llave = "" 
ruta_descargas = os.path.join(os.path.expanduser("~"), "Downloads", nombre_archivo_llave)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = ruta_descargas

# Rutas locales
POPPLER_PATH = r''
DIR_PDFS = r''
client = vision.ImageAnnotatorClient()

def motor_v21_final_completo(path_pdf):
    try:
        # Convertimos las 3 páginas del PDF
        paginas = convert_from_path(path_pdf, dpi=300, first_page=1, last_page=3, poppler_path=POPPLER_PATH)
        
        texto_paginas = []
        palabras_totales = []

        for i, pg in enumerate(paginas):
            img_byte_arr = io.BytesIO()
            pg.save(img_byte_arr, format='JPEG', quality=95)
            image = vision.Image(content=img_byte_arr.getvalue())
            response = client.text_detection(image=image)
            
            if response.text_annotations:
                texto_paginas.append(response.text_annotations[0].description.upper())
                for p in response.text_annotations[1:]:
                    palabras_totales.append({'pals': p, 'pag': i+1})
            else:
                texto_paginas.append("")

        texto_p1 = texto_paginas[0] if len(texto_paginas) >= 1 else ""
        texto_p2 = texto_paginas[1] if len(texto_paginas) >= 2 else ""
        texto_p3 = texto_paginas[2] if len(texto_paginas) >= 3 else ""
        texto_global = " ".join(texto_paginas).replace("\n", " ")

        # --- HERRAMIENTA: BÚSQUEDA POR COORDENADAS ---
        def obtener_por_coordenada(etiquetas, pag_objetivo, margen_y=35):
            y_r, x_r = -1, -1
            for item in palabras_totales:
                if item['pag'] == pag_objetivo:
                    if any(et.upper() in item['pals'].description.upper() for et in etiquetas):
                        y_r = item['pals'].bounding_poly.vertices[0].y
                        x_r = item['pals'].bounding_poly.vertices[1].x
                        break
            if y_r == -1: return ""
            linea = [item['pals'].description for item in palabras_totales 
                     if item['pag'] == pag_objetivo and abs(item['pals'].bounding_poly.vertices[0].y - y_r) < margen_y 
                     and item['pals'].bounding_poly.vertices[0].x > (x_r - 2)]
            res = " ".join(linea).strip()
            for et in etiquetas: res = res.replace(et.upper(), "")
            return res.replace(":", "").replace("|", "").strip()

        # --- LÓGICA DE FOLIO (BLINDADA) ---
        def extraer_folio_seguro():
            # Buscamos primero en P1 (es la fuente más limpia para el folio)
            candidatos = re.findall(r'\b\d{8}\b', texto_p1)
            # Si no hay en P1, buscamos en P3 pero con filtro estricto
            if not candidatos:
                candidatos = re.findall(r'\b\d{8}\b', texto_p3)
            
            for f in candidatos:
                # El folio no empieza con 20 (años) ni 19
                if not f.startswith(('20', '19')):
                    return f
            return candidatos[0] if candidatos else "REVISAR"

        # --- LÓGICA DE TRÁMITE (PÁGINA 3) ---
        def extraer_tramite_p3():
            val_p3 = obtener_por_coordenada(["TRAMITE", "TIPO"], 3)
            if "RENOV" in val_p3 or "RENOVACION" in texto_p3: return "RENOVACION"
            if "DUPLIC" in val_p3 or "DUPLICADO" in texto_p3: return "DUPLICADO"
            if "PERMIS" in val_p3 or "PERMISO" in texto_p3: return "PERMISO"
            return "NUEVA"

        # --- CONSTRUCCIÓN DEL DICCIONARIO FINAL ---
        return {
            "Folio": extraer_folio_seguro(),
            "Nombre": obtener_por_coordenada(["NOMBRE", "TITULAR"], 2),
            "CURP": (re.search(r'[A-Z]{4}\d{6}[A-Z]{6}[A-Z0-9]{2}', texto_global.replace(" ", "")) or re.search(r'', '')).group(0),
            "Sexo": "MUJER" if "MUJER" in texto_global else ("HOMBRE" if "HOMBRE" in texto_global else ""),
            "Fecha_expedicion": (re.search(r'\d{2}/\d{2}/\d{4}', texto_p3 if texto_p3 else texto_p1) or re.search(r'', '')).group(0),
            "Modulo": "TESORERIA EXPRESS MIGUEL ANGEL DE QUEVEDO",
            "Vigencia": obtener_por_coordenada(["VIGENCIA"], 3) or "PERMANENTE",
            "Tipo_tramite": extraer_tramite_p3()
        }

    except Exception as e:
        print(f"Error en archivo: {e}")
        return None

# --- 2. PROCESAMIENTO MASIVO ---
if __name__ == "__main__":
    archivos = [f for f in os.listdir(DIR_PDFS) if f.lower().endswith(".pdf")]
    resultados = []
    
    print(f"🚀 Iniciando V21: Extracción Total (P3: Trámite/Vigencia | P2: Nombre | P1: Folio)")
    
    for i, f in enumerate(archivos, 1):
        info = motor_v21_final_completo(os.path.join(DIR_PDFS, f))
        if info:
            resultados.append({"Archivo": f, **info})
        if i % 10 == 0:
            print(f"📊 Procesados: {i}/{len(archivos)}...")

    # Guardar en CSV
    df = pd.DataFrame(resultados)
    columnas_orden = ["Archivo", "Folio", "Nombre", "CURP", "Sexo", "Fecha_expedicion", "Modulo", "Vigencia", "Tipo_tramite"]
    
    ruta_csv = os.path.join(DIR_PDFS, "BASE_DATOS_V21_EXITO.csv")
    df[columnas_orden].to_csv(ruta_csv, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ Proceso terminado con éxito.")
    print(f"📂 Archivo guardado en: {ruta_csv}")